# 🧹 Step 1.2 — Preprocessing Pipeline (VLSP 2018)

**Mục tiêu:**
- Refactor logic preprocessing thành module `src/data/preprocessor.py`
- Tokenization tiếng Việt (underthesea → pyvi fallback)
- Unicode NFC, xử lý URL/emoji/lặp ký tự
- Tính class weights cho imbalanced aspects
- Train/Val/Test split với stratify
- Lưu parquet vào `data/processed/`

In [ ]:
import os, sys, pathlib, subprocess, importlib
root = pathlib.Path('..').resolve()
sys.path.insert(0, str(root))

def _safe_import(mod: str, pkg: str | None = None) -> bool:
    """Trả về True nếu import được (không lỗi Import/OS/DLL)."""
    pkg = pkg or mod
    try:
        importlib.import_module(mod)
        return True
    except ImportError:
        print(f'📦 Cài {pkg}...')
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
            importlib.import_module(mod)
            return True
        except Exception as e:
            print(f'   ⚠️  Không cài/import được {pkg}: {e}')
            return False
    except OSError as e:
        # Ví dụ: torch DLL lỗi → underthesea không chạy được
        print(f'   ⚠️  {mod} có lỗi runtime/DLL: {e.__class__.__name__}: {str(e)[:120]}')
        return False

# Bắt buộc
for mod, pkg in [('pandas', 'pandas'), ('numpy', 'numpy'),
                 ('sklearn', 'scikit-learn'), ('pyarrow', 'pyarrow'),
                 ('matplotlib', 'matplotlib'), ('seaborn', 'seaborn')]:
    _safe_import(mod, pkg)

# Tokenizer tiếng Việt — ưu tiên underthesea, fallback pyvi
has_underthesea = _safe_import('underthesea', 'underthesea')
has_pyvi        = _safe_import('pyvi', 'pyvi')

if not has_underthesea:
    # Báo cho preprocessor biết để skip underthesea (tránh re-raise OSError torch)
    os.environ['DISABLE_UNDERTHESEA'] = '1'
    print('ℹ️  Sẽ dùng pyvi (underthesea bị vô hiệu do lỗi DLL/torch).')

if not (has_underthesea or has_pyvi):
    raise RuntimeError(
        "❌ Không có tokenizer tiếng Việt nào khả dụng.\n"
        "   Cài bằng: pip install pyvi"
    )

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from IPython.display import display   # ⬅ FIX: thiếu import gây NameError ở cell class weights

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
sns.set_theme(style='whitegrid')

from src.data.preprocessor import VLSPPreprocessor, PreprocessConfig
print('✅ Import thành công')

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\annt43\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

## 1. Cấu hình & Scan toàn bộ data

In [ ]:
# Dựa trên EDA (Step 1.1): median ~40 từ, pct_over_128 thấp → max_words=200 ổn
cfg = PreprocessConfig(
    lowercase       = True,
    nfc_normalize   = True,
    remove_url      = True,
    remove_email    = True,
    remove_emoji    = False,   # giữ emoji (hữu ích cho sentiment)
    collapse_repeat = True,
    tokenize        = True,
    min_words       = 2,
    max_words       = 200,
    random_state    = 42,
)

pp = VLSPPreprocessor(cfg)

# --- Quét TOÀN BỘ data/ để tìm nguyên nhân nếu load sai file ---
search_roots = [pathlib.Path('../data/raw'), pathlib.Path('../data')]
all_files = []
for d in search_roots:
    if d.exists():
        all_files += [f for f in d.rglob('*') if f.suffix.lower() in ('.csv', '.txt', '.tsv', '.json', '.jsonl')]

# Khử trùng lặp, sort
all_files = sorted(set(all_files), key=lambda p: str(p).lower())

print(f"🔎 Tìm thấy {len(all_files)} file dữ liệu:")
for f in all_files:
    try:
        size_kb = f.stat().st_size / 1024
    except OSError:
        size_kb = -1
    print(f"   • {f.relative_to(root) if root in f.parents else f}   ({size_kb:,.1f} KB)")

if not all_files:
    raise FileNotFoundError(
        "❌ Không tìm thấy file dữ liệu nào trong ../data. "
        "Hãy đặt file VLSP vào data/raw/ (ví dụ: 1-VLSP2018-SA-Restaurant-train.txt)."
    )

# --- Ưu tiên file 'train' theo pattern VLSP chuẩn ---
def _score(p: pathlib.Path) -> tuple:
    name = p.name.lower()
    is_train   = 'train' in name
    is_vlsp    = 'vlsp' in name
    is_rest    = 'restaurant' in name or 'hotel' in name
    is_csv     = p.suffix.lower() == '.csv'
    not_test   = 'test' not in name and 'dev' not in name
    # càng cao càng ưu tiên
    return (is_train, not_test, is_vlsp, is_rest, is_csv, -len(name))

train_candidates = sorted(all_files, key=_score, reverse=True)
train_path = train_candidates[0]
print(f"\n📂 Đã chọn file train: {train_path.name}")
print(f"    (dùng biến `train_path` để override nếu chọn nhầm)")

df_raw = pp.load(train_path)
print(f"\n✅ Loaded shape: {df_raw.shape}")
df_raw.head(3)

## 2. Detect schema & Transform

In [ ]:
review_col, aspect_cols = pp.detect_schema(df_raw)
print(f"✅ review_col : {review_col}")
print(f"✅ aspects ({len(aspect_cols)}): {aspect_cols}")

# Sanity check — nguyên nhân thường gặp: detect nhầm cột
assert review_col in df_raw.columns, f"review_col '{review_col}' không có trong df_raw"
assert len(aspect_cols) > 0, "❌ Không phát hiện aspect column nào — kiểm tra lại schema file!"
n_empty = df_raw[review_col].isna().sum() + (df_raw[review_col].astype(str).str.strip() == '').sum()
print(f"ℹ️  Số review rỗng/NaN: {n_empty} / {len(df_raw)}")

In [ ]:
df = pp.transform(df_raw)
summary = pp.summary(df)

print("=== SUMMARY ===")
for k, v in summary.items():
    print(f"  {k:<15s}: {v}")

if len(df) == 0:
    raise RuntimeError("❌ df rỗng sau transform — kiểm tra lại min_words/max_words trong cfg.")

df[[review_col, 'text_clean', 'text_tokenized', 'word_count']].head(5)

## 3. Kiểm tra chất lượng sau preprocessing

In [ ]:
# So sánh độ dài trước/sau (FIX: drop NaN trước khi split)
raw_lens = (df_raw[review_col].dropna().astype(str).str.split().str.len()).values
aft_lens = df['word_count'].values

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw_lens, bins=40, color='#90A4AE', alpha=0.8, label='raw')
axes[0].hist(aft_lens, bins=40, color='#26A69A', alpha=0.8, label='after')
axes[0].set_xlabel('Số từ'); axes[0].set_ylabel('Count')
axes[0].set_title('Word count: raw vs after preprocessing')
axes[0].legend()

sample = df.sample(min(3, len(df)), random_state=42) if len(df) else df.iloc[0:0]
for i, (_, row) in enumerate(sample.iterrows()):
    print(f"\n--- Mẫu {i+1} ---")
    print(f"RAW  : {str(row[review_col])[:120]}")
    print(f"CLEAN: {str(row['text_clean'])[:120]}")
    print(f"TOKEN: {str(row['text_tokenized'])[:120]}")

axes[1].axis('off')
axes[1].text(0.02, 0.95, f"Tokenizer: {summary.get('tokenizer','?')}\n"
                        f"Format   : {summary.get('format_type','?')}\n"
                        f"Rows     : {summary.get('n_rows',0):,}\n"
                        f"Median   : {summary.get('median_words','?')} từ\n"
                        f">128 từ  : {summary.get('pct_over_128',0):.1%}",
             transform=axes[1].transAxes, fontsize=12, family='monospace', va='top')

plt.tight_layout()
plt.show()

## 4. Class weights (xử lý imbalance từ EDA)

In [ ]:
weights = pp.compute_class_weights(df)

rows = [{'aspect': asp, **w} for asp, w in weights.items()]
w_df = pd.DataFrame(rows).set_index('aspect').fillna(0).round(3)

print("=== CLASS WEIGHTS (balanced) ===")
display(w_df)

fig, ax = plt.subplots(figsize=(7, max(3, len(w_df) * 0.5)))
sns.heatmap(w_df, annot=True, fmt='.2f', cmap='YlOrRd', linewidths=0.5, ax=ax)
ax.set_title('Class Weights per Aspect (dùng cho Focal/Weighted Loss)', fontweight='bold')
plt.tight_layout()
out_dir = pathlib.Path('../outputs/preprocessing')
out_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(out_dir / 'class_weights.png')
plt.show()

## 5. Train / Val / Test split

In [ ]:
# Chọn cột stratify
if pp.format_type == 'dense':
    df['_strat'] = df[aspect_cols].isin(['positive', 'negative']).sum(axis=1).clip(0, 5)
    strat_col = '_strat'
else:
    strat_col = max(aspect_cols, key=lambda c: df[c].notna().sum())

# FIX: stratify yêu cầu mỗi class ≥ 2 mẫu. Nếu không đạt → fallback không stratify.
vc = df[strat_col].fillna('__NA__').value_counts()
if (vc < 2).any():
    print(f"⚠️  '{strat_col}' có class < 2 mẫu → tắt stratify. Value counts:\n{vc}")
    strat_col_safe = None
else:
    strat_col_safe = strat_col
    print(f"📊 Stratify theo: {strat_col}")

try:
    splits = pp.split(df, test_size=0.15, val_size=0.15, stratify_on=strat_col_safe)
except ValueError as e:
    print(f"⚠️  Stratify lỗi ({e}) → chuyển sang split ngẫu nhiên.")
    splits = pp.split(df, test_size=0.15, val_size=0.15, stratify_on=None)

print("\n=== SPLIT SIZES ===")
for name, part in splits.items():
    print(f"  {name:<6s}: {len(part):>6,} rows ({len(part)/len(df):.1%})")

In [ ]:
# Verify stratify
if pp.format_type != 'dense' and strat_col in df.columns:
    asp = strat_col
    print(f"--- Phân phối nhãn '{asp}' giữa các split ---")
    for name, part in splits.items():
        vc = part[asp].value_counts(normalize=True, dropna=False).round(3)
        print(f"  {name}: {vc.to_dict()}")

## 6. Lưu kết quả

In [ ]:
import json
from dataclasses import asdict

processed_dir = pathlib.Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

# Bỏ cột tạm
for name in list(splits.keys()):
    if '_strat' in splits[name].columns:
        splits[name] = splits[name].drop(columns=['_strat'])

pp.save_splits(splits, processed_dir)

# Class weights
weights_path = processed_dir / 'class_weights.json'
with open(weights_path, 'w', encoding='utf-8') as fh:
    json.dump(weights, fh, ensure_ascii=False, indent=2)

# Metadata — dùng asdict để an toàn với dataclass
try:
    cfg_dict = asdict(cfg)
except TypeError:
    cfg_dict = dict(cfg.__dict__)

meta = {
    'source_file':  str(train_path.name),
    'review_col':   pp.review_col,
    'aspect_cols':  pp.aspect_cols,
    'format_type':  pp.format_type,
    'tokenizer':    pp.tokenizer_name,
    'config':       cfg_dict,
    'n_rows':       {k: len(v) for k, v in splits.items()},
}
meta_path = processed_dir / 'metadata.json'
with open(meta_path, 'w', encoding='utf-8') as fh:
    json.dump(meta, fh, ensure_ascii=False, indent=2)

print(f"✅ Saved to {processed_dir.resolve()}")
print(f"   • train.parquet / val.parquet / test.parquet")
print(f"   • class_weights.json")
print(f"   • metadata.json")

---
## ✅ Checklist Step 1.2

- [x] Tách logic thành `src/data/preprocessor.py`
- [x] Unicode NFC + clean URL/email/lặp ký tự
- [x] Tokenization tiếng Việt (underthesea / pyvi fallback)
- [x] Chuẩn hóa labels `positive/negative/neutral` → NaN
- [x] Tính `class_weights` cho imbalanced aspects
- [x] Train/Val/Test split (70/15/15) với stratify (auto fallback khi class < 2)
- [x] Lưu Parquet + metadata vào `data/processed/`
- [x] **Diagnostic scan** toàn bộ `data/` + scoring chọn file train đúng

**Bước tiếp theo:** Step 2.1 — Baseline Models → `notebooks/03_Baseline_Models.ipynb`